In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext

appName = "Regresi di Spark"
spark = SparkSession \
    .builder \
    .appName(appName)\
    .config("spark.some.config.option","some-value")\
    .getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
irisDataFrame = spark.read.csv('/content/drive/MyDrive/dataset/Iklan.csv',inferSchema=True, header=True)
irisDataFrame.show()

+---+---------+----+-----+---------+
|_c0|Handphone|  TV|Radio|Penjualan|
+---+---------+----+-----+---------+
|  1|    230.1|37.8| 69.2|     22.1|
|  2|     44.5|39.3| 45.1|     10.4|
|  3|     17.2|45.9| 69.3|      9.3|
|  4|    151.5|41.3| 58.5|     18.5|
|  5|    180.8|10.8| 58.4|     12.9|
|  6|      8.7|48.9| 75.0|      7.2|
|  7|     57.5|32.8| 23.5|     11.8|
|  8|    120.2|19.6| 11.6|     13.2|
|  9|      8.6| 2.1|  1.0|      4.8|
| 10|    199.8| 2.6| 21.2|     10.6|
| 11|     66.1| 5.8| 24.2|      8.6|
| 12|    214.7|24.0|  4.0|     17.4|
| 13|     23.8|35.1| 65.9|      9.2|
| 14|     97.5| 7.6|  7.2|      9.7|
| 15|    204.1|32.9| 46.0|     19.0|
| 16|    195.4|47.7| 52.9|     22.4|
| 17|     67.8|36.6|114.0|     12.5|
| 18|    281.4|39.6| 55.8|     24.4|
| 19|     69.2|20.5| 18.3|     11.3|
| 20|    147.3|23.9| 19.1|     14.6|
+---+---------+----+-----+---------+
only showing top 20 rows



In [ ]:
data = irisDataFrame.select("Handphone","TV","Radio","Penjualan")
data.show(3)

+---------+----+-----+---------+
|Handphone|  TV|Radio|Penjualan|
+---------+----+-----+---------+
|    230.1|37.8| 69.2|     22.1|
|     44.5|39.3| 45.1|     10.4|
|     17.2|45.9| 69.3|      9.3|
+---------+----+-----+---------+
only showing top 3 rows



In [ ]:
dataTerpisahkan = data.randomSplit([0.7,0.3],1)
trainingData = dataTerpisahkan[0]
testingData = dataTerpisahkan[1]
train_rows = trainingData.count()
test_rows = testingData.count()
print("Training :",train_rows,"Testing",test_rows)
trainingData.show(train_rows)
testingData.show(test_rows)

Training : 145 Testing 55
+---------+----+-----+---------+
|Handphone|  TV|Radio|Penjualan|
+---------+----+-----+---------+
|      0.7|39.6|  8.7|      1.6|
|      4.1|11.6|  5.7|      3.2|
|      5.4|29.9|  9.4|      5.3|
|      7.3|28.1| 41.4|      5.5|
|      8.6| 2.1|  1.0|      4.8|
|      8.7|48.9| 75.0|      7.2|
|     11.7|36.9| 45.2|      7.3|
|     13.1| 0.4| 25.6|      5.3|
|     13.2|15.9| 49.6|      5.6|
|     16.9|43.7| 89.4|      8.7|
|     17.2| 4.1| 31.6|      5.9|
|     19.6|20.1| 17.0|      7.6|
|     23.8|35.1| 65.9|      9.2|
|     25.0|11.0| 29.7|      7.2|
|     25.1|25.7| 43.3|      8.5|
|     25.6|39.0|  9.3|      9.5|
|     26.8|33.0| 19.3|      8.8|
|     27.5| 1.6| 20.7|      6.9|
|     28.6| 1.5| 33.0|      7.3|
|     31.5|24.6|  2.2|      9.5|
|     36.9|38.6| 65.6|     10.8|
|     38.0|40.3| 11.9|     10.9|
|     39.5|41.1|  5.8|     10.8|
|     43.0|25.9| 20.5|      9.6|
|     44.7|25.8| 20.6|     10.1|
|     50.0|11.6| 18.4|      8.4|
|     53.5| 2.0| 

In [ ]:
assembler = VectorAssembler(inputCols=["Handphone","TV","Radio"],outputCol="features")

trainingDataFinal = assembler.transform(trainingData).select(
    col("features"),(col("Penjualan").cast("Int").alias("label")))
trainingDataFinal.show(truncate=False, n=12)

+----------------+-----+
|features        |label|
+----------------+-----+
|[0.7,39.6,8.7]  |1    |
|[4.1,11.6,5.7]  |3    |
|[5.4,29.9,9.4]  |5    |
|[7.3,28.1,41.4] |5    |
|[8.6,2.1,1.0]   |4    |
|[8.7,48.9,75.0] |7    |
|[11.7,36.9,45.2]|7    |
|[13.1,0.4,25.6] |5    |
|[13.2,15.9,49.6]|5    |
|[16.9,43.7,89.4]|8    |
|[17.2,4.1,31.6] |5    |
|[19.6,20.1,17.0]|7    |
+----------------+-----+
only showing top 12 rows



In [ ]:

algoritma = LinearRegression(
    labelCol ="label",featuresCol="features",maxIter=10,regParam = 0.3)

model = algoritma.fit(trainingDataFinal)
print("Model regresi selesai ditraining!")
model.coefficients
print("Nilai koefisien regresi = "+str(model.coefficients))
print("Nilai konstanta regresi = "+str(model.intercept))

Model regresi selesai ditraining!
Nilai koefisien regresi = [0.04165914630250659,0.1722656085821103,0.008982479079541741]
Nilai konstanta regresi = 3.092518297260064


In [ ]:
testingDataFinal = assembler.transform(
    testingData).select(
    col("features"),(col("Penjualan")).cast("Int").alias("trueLabel"))
testingDataFinal.show(truncate=False, n=7)

+----------------+---------+
|features        |trueLabel|
+----------------+---------+
|[7.8,38.9,50.6] |6        |
|[8.4,27.2,2.1]  |5        |
|[17.2,45.9,69.3]|9        |
|[17.9,37.6,21.6]|8        |
|[18.7,12.1,23.4]|6        |
|[18.8,21.7,50.4]|7        |
|[19.4,16.0,22.3]|6        |
+----------------+---------+
only showing top 7 rows



In [ ]:
prediksiMentah = model.transform(testingDataFinal)

prediksiFinal = prediksiMentah.select("features","prediction","trueLabel")
prediksiFinal.show(7)

+----------------+------------------+---------+
|        features|        prediction|trueLabel|
+----------------+------------------+---------+
| [7.8,38.9,50.6]|10.573105253688517|        6|
|  [8.4,27.2,2.1]| 8.146942885701558|        5|
|[17.2,45.9,69.3]|12.338532847794282|        9|
|[17.9,37.6,21.6]| 10.50942544688038|        8|
|[18.7,12.1,23.4]| 6.166148207421749|        6|
|[18.8,21.7,50.4]| 8.066590899587885|        7|
|[19.4,16.0,22.3]| 6.857264756316237|        6|
+----------------+------------------+---------+
only showing top 7 rows



## 1. Mencari nilai RMSE


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol = "trueLabel",predictionCol="prediction",metricName="rmse")

rmse = evaluator.evaluate(prediksiFinal)
print("Root Mean Square Error (RMSE):",rmse)

Root Mean Square Error (RMSE): 1.7720613211659475


## 2. Mencari nilai MAE

In [ ]:

from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol = "trueLabel",predictionCol="prediction",metricName="mae")

mae = evaluator.evaluate(prediksiFinal)
print("Mean Absolut Error (MAE):",mae)

Mean Absolut Error (MAE): 1.435299693751245


## 3. Mencari Nilai MSE

In [ ]:

from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol = "trueLabel",predictionCol="prediction",metricName="mse")

mse = evaluator.evaluate(prediksiFinal)
print("Mean Square Error (MSE):",mse)

Mean Square Error (MSE): 3.140201325972403


## 4. Mencari Nilai R2/R-square

In [ ]:

from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol="trueLabel", predictionCol="prediction", metricName="r2")

r2 = evaluator.evaluate(prediksiFinal)
print("R-squared (R2):", r2)


R-squared (R2): 0.9059177444776805
